# Emotion Voice — Train & Export Model (Kaggle / Colab)

Cloud-ready version of `Ai.ipynb`. Produces the exact file the backend loads:
`best_efficientnet_v2_m+vgg16+densenet121.pt`

**Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On** → *Save Version → Save & Run All*.
When the run finishes, open the version's **Output** tab and download the `.pt` file.

**Colab:** Runtime → Change runtime type → **T4 GPU** → Run all. The file is downloaded at the end.

Differences from `Ai.ipynb`: datasets are downloaded automatically, SAVEE surprise/sad labelling bug fixed,
GAN augmentation skipped (class weights + weighted sampler handle imbalance), only the 3 ensemble models are trained.

In [ ]:
!pip install -q noisereduce kagglehub

import os, re, glob, json, copy, random, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import torchaudio, librosa, noisereduce as nr
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
MEL_DIR = os.path.join(WORK, "mel_npy")
CKPT_DIR = os.path.join(WORK, "training_outputs")
EXPORT_DIR = os.path.join(WORK, "final_export")
for d in (MEL_DIR, CKPT_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)

EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']  # must match backend CLASSES

# Audio settings — identical to backend/utils/audio_preprocess.py
SAMPLE_RATE = 22050
TARGET_SAMPLES = int(SAMPLE_RATE * 4.0)
N_MELS, N_FFT, HOP_LENGTH, DESIRED_TIME_FRAMES = 128, 1024, 256, 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if device.type != "cuda":
    print("WARNING: no GPU — enable a GPU accelerator or training will take days.")

## 1. Download datasets (CREMA-D, RAVDESS, SAVEE, TESS)

In [ ]:
import kagglehub

DATASETS = {
    "cremad":  "ejlok1/cremad",
    "ravdess": "uwrfkaggler/ravdess-emotional-speech-audio",
    "savee":   "ejlok1/surrey-audiovisual-expressed-emotion-savee",
    "tess":    "ejlok1/toronto-emotional-speech-set-tess",
}
roots = {k: kagglehub.dataset_download(v) for k, v in DATASETS.items()}
for k, v in roots.items():
    print(f"{k:8s} {v}  ({len(glob.glob(os.path.join(v, '**', '*.wav'), recursive=True))} wav)")

## 2. Label files

In [ ]:
CREMA = {'ANG': 'angry', 'DIS': 'disgust', 'FEA': 'fear', 'HAP': 'happy', 'NEU': 'neutral', 'SAD': 'sad'}
RAVDESS = {'01': 'neutral', '02': 'neutral', '03': 'happy', '04': 'sad',
           '05': 'angry', '06': 'fear', '07': 'disgust', '08': 'surprise'}
SAVEE = {'a': 'angry', 'd': 'disgust', 'f': 'fear', 'h': 'happy', 'n': 'neutral',
         'sa': 'sad', 'su': 'surprise'}

def wavs(root):
    return glob.glob(os.path.join(root, "**", "*.wav"), recursive=True)

items = {}  # unique key -> (path, emotion); dedupes datasets shipped twice in nested folders

for f in wavs(roots["cremad"]):
    parts = Path(f).stem.split("_")
    if len(parts) >= 3 and parts[2] in CREMA:
        items["crema_" + Path(f).name] = (f, CREMA[parts[2]])

for f in wavs(roots["ravdess"]):
    parts = Path(f).stem.split("-")
    if len(parts) == 7 and parts[2] in RAVDESS:
        items["rav_" + Path(f).name] = (f, RAVDESS[parts[2]])

for f in wavs(roots["savee"]):
    m = re.match(r"^([A-Z]{2})_(sa|su|a|d|f|h|n)\d+\.wav$", Path(f).name)
    if m:
        items["savee_" + Path(f).name] = (f, SAVEE[m.group(2)])

for f in wavs(roots["tess"]):
    folder = Path(f).parent.name.lower()
    emo = next((e for e in EMOTIONS if e in folder), None)
    if emo:
        items["tess_" + Path(f).name.lower()] = (f, emo)

print("Total files:", len(items))
print(pd.Series([e for _, e in items.values()]).value_counts().sort_index())

## 3. Clean audio → 128×128 mel spectrograms

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

def safe_audio(y):
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    rms = np.sqrt(np.mean(y ** 2))
    return y / (rms + 1e-8) if rms > 0 else y

def pad_or_trim(y):
    return y[:TARGET_SAMPLES] if len(y) > TARGET_SAMPLES else np.pad(y, (0, TARGET_SAMPLES - len(y)))

def process(key, path, emotion):
    out = os.path.join(MEL_DIR, emotion, key.replace(".wav", ".npy"))
    if os.path.exists(out):
        return True
    import warnings
    warnings.filterwarnings("ignore")  # worker processes don't inherit the notebook's filters
    try:
        y, sr = librosa.load(path, sr=SAMPLE_RATE)
        y = safe_audio(y)
        try:
            # silent stretches trigger a harmless divide-by-zero warning; safe_audio() cleans the NaNs
            y = nr.reduce_noise(y=y, sr=sr, prop_decrease=1.0)
        except Exception:
            pass
        y = pad_or_trim(safe_audio(y))
        with torch.no_grad():
            mel_db = amp_to_db(mel_transform(torch.tensor(y, dtype=torch.float32).unsqueeze(0)))
            mel = F.interpolate(mel_db.unsqueeze(1), size=(N_MELS, DESIRED_TIME_FRAMES),
                                mode="bilinear", align_corners=False)
        np.save(out, mel.squeeze().numpy().astype(np.float32))
        return True
    except Exception as e:
        print("skip", path, e)
        return False

for e in EMOTIONS:
    os.makedirs(os.path.join(MEL_DIR, e), exist_ok=True)

torch.set_num_threads(1)
# Takes ~20-40 min on Kaggle's 4 CPUs; re-running resumes (finished files are skipped)
jobs = Parallel(n_jobs=os.cpu_count(), return_as="generator_unordered")(
    delayed(process)(k, p, e) for k, (p, e) in items.items())
results = list(tqdm(jobs, total=len(items), desc="Preprocessing", mininterval=10))
torch.set_num_threads(os.cpu_count())
print(f"OK: {sum(results)} / {len(results)}")

## 4. Dataset & split (same 80/20 stratified split, seed 42)

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def find_classes_and_files(root):
    classes = sorted(d.name for d in Path(root).iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    files, labels = [], []
    for c in classes:
        for f in sorted(Path(root, c).glob("*.npy")):
            files.append(str(f)); labels.append(class_to_idx[c])
    return classes, class_to_idx, files, labels

class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files, self.labels = files, labels
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        mel = np.load(self.files[idx]).astype(np.float32)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return torch.tensor(mel).unsqueeze(0), torch.tensor(self.labels[idx])

classes, _, files, labels = find_classes_and_files(MEL_DIR)
assert classes == EMOTIONS, classes
n_classes = len(classes)

sss = StratifiedShuffleSplit(1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(files, labels))
train_files, val_files = [files[i] for i in train_idx], [files[i] for i in val_idx]
train_labels, val_labels = [labels[i] for i in train_idx], [labels[i] for i in val_idx]

label_counts = Counter(train_labels)
total = sum(label_counts.values())
class_weight_tensor = torch.tensor([total / (n_classes * label_counts[i]) for i in range(n_classes)],
                                   dtype=torch.float32).to(device)
sampler = WeightedRandomSampler([1.0 / label_counts[l] for l in train_labels],
                                len(train_labels), replacement=True)

BATCH_SIZE = 32
NUM_WORKERS = 2
train_loader = DataLoader(MelDataset(train_files, train_labels), batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(MelDataset(val_files, val_labels), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
print(f"train={len(train_files)}  val={len(val_files)}")

## 5. Train EfficientNetV2-M, VGG16, DenseNet121 (Phase 2 settings)

In [ ]:
EPOCHS = 40
LR = 2e-4
PATIENCE = 6
L1_LAMBDA = 1e-5
L2_WEIGHT_DECAY = 1e-4
MIXED_PRECISION = device.type == "cuda"

def build_model(name, n_classes, pretrained=True):
    if name == "efficientnet_v2_m":
        m = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.IMAGENET1K_V1 if pretrained else None)
        m.features[0][0] = nn.Conv2d(1, m.features[0][0].out_channels, 3, 2, 1, bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        return m, optim.AdamW
    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        m.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        m.classifier = nn.Linear(m.classifier.in_features, n_classes)
        return m, optim.Adam
    if name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        m.features[0] = nn.Conv2d(1, 64, 3, padding=1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, n_classes)
        return m, lambda params, lr, **kw: optim.SGD(params, lr=lr, momentum=0.9, **kw)
    raise ValueError(name)

def evaluate(model, loader, criterion):
    model.eval(); preds, trues, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device.type, enabled=MIXED_PRECISION):
                out = model(x)
                loss = criterion(out, y)
            loss_sum += loss.item() * x.size(0)
            preds.extend(out.argmax(1).cpu().numpy()); trues.extend(y.cpu().numpy())
    return loss_sum / len(loader.dataset), accuracy_score(trues, preds), f1_score(trues, preds, average="weighted")

def train(name):
    ckpt = os.path.join(CKPT_DIR, f"best_{name}.pt")
    if os.path.exists(ckpt + ".done"):
        print(f"{name}: already trained, skipping"); return
    print(f"\n{'=' * 70}\nTraining {name}\n{'=' * 70}")
    model, optim_fn = build_model(name, n_classes)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
    optimizer = optim_fn(model.parameters(), lr=LR, weight_decay=L2_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = torch.amp.GradScaler(enabled=MIXED_PRECISION)
    best_f1, bad_epochs = -1, 0

    for epoch in range(1, EPOCHS + 1):
        model.train(); total_loss, n = 0.0, 0
        pbar = tqdm(train_loader, desc=f"[{name}] {epoch}/{EPOCHS}", leave=False)
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.autocast(device.type, enabled=MIXED_PRECISION):
                loss = criterion(model(x), y)
            if L1_LAMBDA > 0:
                loss = loss + L1_LAMBDA * sum(p.abs().sum() for p in model.parameters() if p.requires_grad)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * x.size(0); n += x.size(0)
            pbar.set_postfix(loss=f"{total_loss / n:.4f}")

        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)
        print(f"[{name}] epoch {epoch}: train_loss={total_loss / n:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, bad_epochs = val_f1, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Early stopping (best F1={best_f1:.4f})"); break

    open(ckpt + ".done", "w").write(str(best_f1))
    print(f"{name}: best val F1 = {best_f1:.4f}")
    del model; torch.cuda.empty_cache()

MODEL_NAMES = ["efficientnet_v2_m", "vgg16", "densenet121"]  # order fixes the exported filename
for name in MODEL_NAMES:
    train(name)

## 6. Evaluate ensemble & export TorchScript

In [ ]:
class Ensemble(nn.Module):
    def __init__(self, models_):
        super().__init__(); self.models = nn.ModuleList(models_)
    def forward(self, x):
        return torch.mean(torch.stack([F.softmax(m(x), dim=1) for m in self.models]), dim=0)

loaded = []
for name in MODEL_NAMES:
    m, _ = build_model(name, n_classes, pretrained=False)
    m.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"best_{name}.pt"), map_location="cpu"))
    loaded.append(m.float().eval())

ensemble = Ensemble(loaded).to(device).eval()
probs, trues = [], []
with torch.no_grad():
    for x, y in tqdm(val_loader, desc="Ensemble eval"):
        probs.append(ensemble(x.to(device)).cpu()); trues.extend(y.numpy())
preds = torch.cat(probs).argmax(1).numpy()
print(f"Ensemble val acc={accuracy_score(trues, preds):.4f}  f1={f1_score(trues, preds, average='weighted'):.4f}")

ensemble = ensemble.to("cpu").float().eval()
out_path = os.path.join(EXPORT_DIR, "best_" + "+".join(MODEL_NAMES) + ".pt")
with torch.no_grad():
    traced = torch.jit.trace(ensemble, torch.randn(1, 1, 128, 128))
traced.save(out_path)

# sanity check: reload and confirm output is a probability vector
reloaded = torch.jit.load(out_path)
p = reloaded(torch.randn(1, 1, 128, 128))
assert p.shape == (1, 7) and abs(p.sum().item() - 1) < 1e-3
print(f"Saved {out_path}  ({os.path.getsize(out_path) / 1e6:.0f} MB)")

In [ ]:
# Colab: download the model. On Kaggle, grab it from the notebook version's Output tab instead.
# (Kaggle ships the google.colab package too, so check for Kaggle explicitly.)
if os.path.isdir("/kaggle/working"):
    print("Kaggle: download from the Output tab ->", out_path)
else:
    from google.colab import files
    files.download(out_path)